In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("10_dropped_variables_and_renamed.csv")

df["week"] = pd.to_datetime(df["week"], dayfirst=True, errors="coerce")

if df["week"].isna().sum() > 0:
    bad_rows = df[df["week"].isna()]
    raise ValueError(
        f"Some dates in 'week' could not be parsed. Check the date format.\n"
        f"Problem rows:\n{bad_rows.head()}"
    )


In [3]:
df = df.sort_values(["coin", "week"]).reset_index(drop=True)

In [4]:
target = "ret_1w"

# Keep all predictors except non-predictor columns
drop_cols = ["week", "ret_1w_original"]
predictor_cols = [col for col in df.columns if col not in drop_cols + [target]]


In [5]:
train_start = pd.Timestamp("2018-01-01")
train_end   = pd.Timestamp("2022-12-30")

valid_start = pd.Timestamp("2023-01-01")
valid_end   = pd.Timestamp("2024-12-30")

test_start  = pd.Timestamp("2025-01-01")
test_end    = pd.Timestamp("2026-01-08")

# Initial estimation sample = train + valid
initial_estimation = df[(df["week"] >= train_start) & (df["week"] <= valid_end)].copy()
test = df[(df["week"] >= test_start) & (df["week"] <= test_end)].copy()

print("Initial estimation sample:", initial_estimation.shape)
print("Test sample:", test.shape)


Initial estimation sample: (4380, 50)
Test sample: (636, 50)


In [7]:
# Helper function to prepare X
#    coin fixed effects via dummies
#    numeric conversion
#    missing value fill
#    alignment to training columns


def prepare_X(data, predictor_cols, fit_columns=None, fill_values=None):
    X = data[predictor_cols].copy()

    # Fixed effects: coin dummies
    if "coin" in X.columns:
        X = pd.get_dummies(X, columns=["coin"], drop_first=True)

    # Convert to numeric
    X = X.apply(pd.to_numeric, errors="coerce")

    # Fill missing values using training medians if supplied
    if fill_values is not None:
        X = X.fillna(fill_values)
    else:
        fill_values = X.median(numeric_only=True)
        X = X.fillna(fill_values)

    # Align columns
    if fit_columns is not None:
        X = X.reindex(columns=fit_columns, fill_value=0)

    # Force float
    X = X.astype(float)

    # Add constant
    X = sm.add_constant(X, has_constant="add")

    return X, fill_values


In [8]:
# 7. Rolling one-step-ahead forecasting
test_weeks = sorted(test["week"].unique())

all_predictions = []

for current_week in test_weeks:
    # Expanding window: use all data up to the week before current_week
    estimation_sample = df[df["week"] < current_week].copy()

    # Current week data to forecast
    current_test = test[test["week"] == current_week].copy()

    if estimation_sample.empty or current_test.empty:
        continue

    # Prepare training X/y
    X_est, train_fill_values = prepare_X(estimation_sample, predictor_cols)
    y_est = pd.to_numeric(estimation_sample[target], errors="coerce").astype(float)

    # Drop rows with missing y if any
    valid_idx = y_est.notna()
    X_est = X_est.loc[valid_idx]
    y_est = y_est.loc[valid_idx]

    # Prepare current test X using training columns and training fill values
    X_curr, _ = prepare_X(
        current_test,
        predictor_cols,
        fit_columns=[c for c in X_est.columns if c != "const"],
        fill_values=train_fill_values
    )
    X_curr = X_curr.reindex(columns=X_est.columns, fill_value=0).astype(float)

    # Fit FE regression
    fe_model = sm.OLS(y_est, X_est).fit()

    # Predict current week
    curr_pred = fe_model.predict(X_curr)

    # Store results
    week_output = pd.DataFrame({
        "coin": current_test["coin"].values,
        "week": current_test["week"].values,
        "actual": current_test[target].values,
        "prediction": curr_pred.values,
    })
    week_output["residual"] = week_output["actual"] - week_output["prediction"]

    all_predictions.append(week_output)

# Combine all weekly forecasts
if len(all_predictions) == 0:
    raise ValueError("No rolling forecasts were generated.")

residual_output = pd.concat(all_predictions, axis=0).reset_index(drop=True)


In [9]:
# overall rmse
overall_rmse = np.sqrt(np.mean((residual_output["actual"] - residual_output["prediction"]) ** 2))
print("\nOverall RMSE:", overall_rmse)



Overall RMSE: 0.12888515803519004


In [10]:
# matching date format 

residual_output["week"] = pd.to_datetime(residual_output["week"]).dt.strftime("%-d/%-m/%Y")

In [11]:
# saving residuals 
residual_output = residual_output[["coin", "week", "actual", "prediction", "residual"]]
residual_output.to_csv("panel_fe_residuals.csv", index=False)
print("Saved: panel_fe_residuals.csv")


Saved: panel_fe_residuals.csv


In [12]:
# saving rmse 

rmse_by_coin = (
    residual_output
    .groupby("coin")
    .apply(lambda g: np.sqrt(np.mean((g["actual"] - g["prediction"]) ** 2)))
    .reset_index(name="rmse")
)

rmse_by_coin.to_csv("panel_fe_RMSE.csv", index=False)
print("Saved: panel_fe_RMSE.csv")

print("\nCoin-specific RMSE:")
print(rmse_by_coin)


Saved: panel_fe_RMSE.csv

Coin-specific RMSE:
    coin      rmse
0    BCH  0.110531
1    BTC  0.078478
2   DASH  0.157937
3    EOS  0.142492
4    ETC  0.113282
5    ETH  0.121803
6   IOTA  0.145757
7    LTC  0.106525
8    OMG  0.117583
9    XMR  0.106338
10   XRP  0.119242
11   ZEC  0.189997


In [13]:
print("\nResidual output preview:")
print(residual_output.head())

print("\nResidual summary:")
print(residual_output["residual"].describe())



Residual output preview:
   coin      week    actual  prediction  residual
0   BCH  5/1/2025  0.074073    0.030688  0.043386
1   BTC  5/1/2025  0.051110    0.014437  0.036673
2  DASH  5/1/2025  0.100478    0.009809  0.090670
3   EOS  5/1/2025  0.160613    0.023952  0.136661
4   ETC  5/1/2025  0.102745    0.035436  0.067309

Residual summary:
count    636.000000
mean       0.001871
std        0.128973
min       -0.380197
25%       -0.075348
50%       -0.004340
75%        0.070293
max        0.473081
Name: residual, dtype: float64
